# 🌿 Dravya Labs — Ayurvedic Herb Model Training (Local)

This notebook trains a PyTorch classification model on the `ayurvedic_herbs.csv` dataset **locally**.

**What it does:**
1. Loads & cleans `ayurvedic_herbs.csv` (Amidha Ayurveda Database — 700+ herbs)
2. Engineers TF-IDF text features (Rasa, Virya, Vipaka, etc.) + numeric features (Doshas)
3. Trains a feedforward neural network on the **full dataset**
4. Exports `herb_model.pth` + `model_metadata.json` + `herb_lookup.csv` to `herbs/model/`


## 1️⃣ Install Dependencies

In [ ]:
!pip install -q torch pandas scikit-learn numpy matplotlib

## 2️⃣ Set Paths & Load CSV

In [ ]:
import os

# Resolve paths relative to this notebook
NOTEBOOK_DIR = os.path.abspath('')  # current working dir when notebook runs
# If running from the training/ directory:
HERBS_DIR = os.path.dirname(NOTEBOOK_DIR) if os.path.basename(NOTEBOOK_DIR) == 'training' else NOTEBOOK_DIR

CSV_FILE = os.path.join(HERBS_DIR, 'ayurvedic_herbs.csv')

# Fallback: try absolute path if relative doesn't work
if not os.path.exists(CSV_FILE):
    CSV_FILE = '/Users/himanshusolanki/Desktop/CU/Dravya-labs2/herbs/ayurvedic_herbs.csv'

if os.path.exists(CSV_FILE):
    file_size = os.path.getsize(CSV_FILE) / 1024
    print(f'✅ Found: {CSV_FILE} ({file_size:.1f} KB)')
else:
    raise FileNotFoundError(f'❌ ayurvedic_herbs.csv not found! Tried: {CSV_FILE}')

## 3️⃣ Load & Clean Data

In [ ]:
import pandas as pd
import numpy as np
import json

TEXT_COLUMNS = [
    "preview", "rasa", "guna", "virya", "vipaka",
    "prabhava", "therapeutic_uses", "category",
    "contraindications", "pacify_dosha", "aggravate_dosha",
]
DOSHA_COLUMNS = [
    "pacify_vata", "pacify_pitta", "pacify_kapha",
    "aggravate_vata", "aggravate_pitta", "aggravate_kapha",
]
NUMERIC_COLUMNS = DOSHA_COLUMNS + ["tridosha_flag"]
ID_COLUMN = "name"
HINDI_COLUMN = "hindi_name"

df = pd.read_csv(CSV_FILE, encoding="utf-8", low_memory=False)
print(f"Raw: {len(df)} rows, {len(df.columns)} columns")

df = df.dropna(subset=[ID_COLUMN])
df = df[df[ID_COLUMN].str.strip() != ""]

for col in TEXT_COLUMNS:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)

if "tridosha" in df.columns:
    df["tridosha_flag"] = df["tridosha"].astype(str).str.lower().map(
        {"true": 1, "1": 1, "false": 0, "0": 0}
    ).fillna(0).astype(int)
else:
    df["tridosha_flag"] = 0

for col in DOSHA_COLUMNS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
    else:
        df[col] = 0

if HINDI_COLUMN in df.columns:
    df[HINDI_COLUMN] = df[HINDI_COLUMN].fillna("")

for col in TEXT_COLUMNS:
    if col in df.columns:
        df[col] = df[col].str.replace(r"\s+", " ", regex=True).str.strip()

before = len(df)
df = df.drop_duplicates(subset=[ID_COLUMN], keep="first").reset_index(drop=True)
print(f"After dedup: {len(df)} (removed {before - len(df)})")
print(f"\n✅ Clean dataset: {len(df)} unique Ayurvedic herbs")

## 4️⃣ Feature Engineering

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

parts = [df[col] for col in TEXT_COLUMNS if col in df.columns]
combined = parts[0]
for p in parts[1:]:
    combined = combined + " " + p
df["combined_text"] = combined.str.replace(r"\s+", " ", regex=True).str.strip()

print(f"Sample combined text (first 200 chars):")
print(df["combined_text"].iloc[0][:200])

TFIDF_MAX_FEATURES = 3000

tfidf = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True,
)

tfidf_matrix = tfidf.fit_transform(df["combined_text"]).toarray()
print(f"\nTF-IDF shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

numeric_cols_present = [c for c in NUMERIC_COLUMNS if c in df.columns]
numeric_data = df[numeric_cols_present].values.astype(np.float32)
print(f"Dosha features shape: {numeric_data.shape}")

X = np.hstack([tfidf_matrix, numeric_data]).astype(np.float32)
INPUT_DIM = X.shape[1]
print(f"\n✅ Final feature matrix: {X.shape}  (input_dim = {INPUT_DIM})")

## 5️⃣ Label Encoding

In [ ]:
unique_names = sorted(df[ID_COLUMN].unique())
name_to_id = {name: idx for idx, name in enumerate(unique_names)}
id_to_name = {idx: name for name, idx in name_to_id.items()}

y = df[ID_COLUMN].map(name_to_id).values
NUM_CLASSES = len(unique_names)

print(f"✅ Classes: {NUM_CLASSES}")
print(f"   Label range: {y.min()} – {y.max()}")

## 6️⃣ Create DataLoader (Full Dataset)

We train on the **entire dataset** since each plant has only 1 sample.
For a retrieval system, the model must learn all plants — not generalize to unseen ones.

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(torch.FloatTensor(X), torch.LongTensor(y))

BATCH_SIZE = 64
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"✅ Full dataset: {len(dataset)} samples")
print(f"   Batches: {len(train_loader)} (batch_size={BATCH_SIZE})")

## 7️⃣ Model Definition

In [ ]:
import torch.nn as nn

class HerbKnowledgeModel(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.network(x)

# Use MPS (Apple Silicon) if available, otherwise CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"🖥️  Device: {device}")

model = HerbKnowledgeModel(INPUT_DIM, NUM_CLASSES).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"📊 Parameters: {total_params:,}")
print(f"   Architecture: {INPUT_DIM} → 512 → 256 → {NUM_CLASSES}")

## 8️⃣ Training Loop

In [ ]:
import time

EPOCHS = 120
LEARNING_RATE = 1e-3
TARGET_ACCURACY = 0.85  # Target 85% accuracy

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

history = {"train_loss": [], "train_acc": []}
best_acc = 0.0

# Save best model directly to herbs/model/
OUTPUT_DIR = os.path.join(HERBS_DIR, 'model')
os.makedirs(OUTPUT_DIR, exist_ok=True)
best_model_path = os.path.join(OUTPUT_DIR, 'herb_model.pth')

print(f"\n🚀 Training for up to {EPOCHS} epochs (target: {TARGET_ACCURACY*100:.0f}% accuracy)")
print(f"📁 Model output: {OUTPUT_DIR}")
print("=" * 70)

for epoch in range(EPOCHS):
    start = time.time()
    model.train()
    total_loss = 0.0
    total_batches = 0
    correct = 0
    total = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1
        _, predicted = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

    scheduler.step()

    avg_loss = total_loss / total_batches
    acc = correct / total
    elapsed = time.time() - start

    history["train_loss"].append(avg_loss)
    history["train_acc"].append(acc)

    marker = ""
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), best_model_path)
        marker = " ✅ best"

    if (epoch + 1) % 5 == 0 or acc > best_acc - 0.001 or marker:
        print(
            f"Epoch {epoch+1:2d}/{EPOCHS} │ "
            f"Loss: {avg_loss:.4f} │ "
            f"Accuracy: {acc*100:.2f}% │ "
            f"{elapsed:.1f}s{marker}"
        )

    if acc >= TARGET_ACCURACY:
        print(f"\n🎯 Target accuracy {TARGET_ACCURACY*100:.0f}% reached at epoch {epoch+1}!")
        break

print(f"\n✅ Training complete! Best accuracy: {best_acc*100:.2f}%")

## 9️⃣ Training Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history["train_loss"], label="Train Loss", linewidth=2, color="#e74c3c")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot([a*100 for a in history["train_acc"]], label="Train Accuracy", linewidth=2, color="#2ecc71")
ax2.axhline(y=80, color='orange', linestyle='--', label='80% Target', alpha=0.7)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Training Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()

# Save training curves to output dir
curves_path = os.path.join(OUTPUT_DIR, 'training_curves.png')
plt.savefig(curves_path, dpi=150)
plt.show()
print(f"✅ Saved {curves_path}")

## 🔟 Model Accuracy Report

Evaluates the best model checkpoint with Top-1, Top-5, and Top-10 accuracy.

In [ ]:
# Load the best model checkpoint
model.load_state_dict(torch.load(best_model_path, weights_only=True))
model.eval()
model.to(device)

correct_top1 = 0
correct_top5 = 0
correct_top10 = 0
total = 0

eval_loader = DataLoader(dataset, batch_size=512, shuffle=False)

with torch.no_grad():
    for batch_X, batch_y in eval_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = model(batch_X)

        # Top-1
        _, pred1 = torch.max(outputs, 1)
        correct_top1 += (pred1 == batch_y).sum().item()

        # Top-5
        _, pred5 = torch.topk(outputs, min(5, NUM_CLASSES), dim=1)
        for i in range(batch_y.size(0)):
            if batch_y[i] in pred5[i]:
                correct_top5 += 1

        # Top-10
        _, pred10 = torch.topk(outputs, min(10, NUM_CLASSES), dim=1)
        for i in range(batch_y.size(0)):
            if batch_y[i] in pred10[i]:
                correct_top10 += 1

        total += batch_y.size(0)

acc_top1 = correct_top1 / total * 100
acc_top5 = correct_top5 / total * 100
acc_top10 = correct_top10 / total * 100

print(f"{'=' * 50}")
print(f"📊 MODEL ACCURACY REPORT")
print(f"{'=' * 50}")
print(f"   Total plants:    {total:,}")
print(f"   Total classes:   {NUM_CLASSES:,}")
print(f"{'─' * 50}")
print(f"   Top-1  Accuracy: {acc_top1:.2f}% ({correct_top1:,}/{total:,})")
print(f"   Top-5  Accuracy: {acc_top5:.2f}% ({correct_top5:,}/{total:,})")
print(f"   Top-10 Accuracy: {acc_top10:.2f}% ({correct_top10:,}/{total:,})")
print(f"{'=' * 50}")

if acc_top1 >= 80:
    print(f"\n✅ Model accuracy is {acc_top1:.1f}% — GOOD for production!")
elif acc_top1 >= 60:
    print(f"\n⚡ Model accuracy is {acc_top1:.1f}% — acceptable, Top-5 covers {acc_top5:.1f}%")
else:
    print(f"\n⚠️  Model accuracy is {acc_top1:.1f}% — consider training for more epochs")

## 1️⃣1️⃣ Export Model Artifacts

In [ ]:
# 1. Model weights already saved during training at best_model_path
print(f"✅ Model: {best_model_path} ({os.path.getsize(best_model_path)/1024/1024:.1f} MB)")

# 2. Save metadata with accuracy
metadata = {
    "num_classes": NUM_CLASSES,
    "input_dim": INPUT_DIM,
    "tfidf_max_features": TFIDF_MAX_FEATURES,
    "tfidf_vocabulary": {k: int(v) for k, v in tfidf.vocabulary_.items()},
    "numeric_columns": numeric_cols_present,
    "text_columns": TEXT_COLUMNS,
    "dosha_columns": DOSHA_COLUMNS,
    "id_to_name": {str(k): v for k, v in id_to_name.items()},
    "name_to_id": name_to_id,
    "dataset": "Amidha Ayurveda Herb Database (700+ herbs)",
    "training_config": {
        "epochs_run": len(history["train_loss"]),
        "best_accuracy": round(best_acc * 100, 2),
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
    },
    "accuracy": {
        "top1": round(acc_top1, 2),
        "top5": round(acc_top5, 2),
        "top10": round(acc_top10, 2),
    },
}

meta_path = os.path.join(OUTPUT_DIR, "model_metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"✅ Metadata: {meta_path} ({os.path.getsize(meta_path)/1024/1024:.1f} MB)")

# 3. Save herb lookup table
lookup_cols = [
    ID_COLUMN, "latin_name", HINDI_COLUMN,
    "preview", "rasa", "guna", "virya", "vipaka",
    "prabhava", "pacify_dosha", "aggravate_dosha",
    "tridosha", "category", "therapeutic_uses",
    "contraindications", "source_url",
]
lookup_cols = [c for c in lookup_cols if c in df.columns]

lookup_path = os.path.join(OUTPUT_DIR, "herb_lookup.csv")
df[lookup_cols].to_csv(lookup_path, index=False, encoding="utf-8")
print(f"✅ Lookup: {lookup_path} ({os.path.getsize(lookup_path)/1024/1024:.1f} MB)")

print(f"\n📦 All artifacts saved to: {OUTPUT_DIR}/")

## 1️⃣2️⃣ Quick Test — Predict from the trained model

In [ ]:
def quick_predict(query: str, top_k: int = 5):
    """Test prediction using the trained model."""
    model.eval()
    model.to("cpu")

    tfidf_feat = tfidf.transform([query]).toarray()
    numeric_pad = np.zeros((1, len(numeric_cols_present)), dtype=np.float32)
    features = np.hstack([tfidf_feat, numeric_pad]).astype(np.float32)
    tensor = torch.FloatTensor(features)

    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)
        top_probs, top_idx = torch.topk(probs, top_k, dim=1)

    print(f"\n🔍 Query: '{query}'")
    print(f"{'Rank':<7}{'Confidence':<13}{'Name':<40}{'Rasa':<20}{'Virya'}")
    print("─" * 95)
    for rank, (prob, idx) in enumerate(zip(top_probs[0], top_idx[0]), 1):
        herb_name = id_to_name[idx.item()]
        row = df[df[ID_COLUMN] == herb_name]
        rasa_val = str(row["rasa"].values[0])[:18] if len(row) > 0 and "rasa" in row.columns else ""
        virya_val = str(row["virya"].values[0])[:18] if len(row) > 0 and "virya" in row.columns else ""
        print(f"  {rank:<5}{prob.item():.6f}     {herb_name:<40}{rasa_val:<20}{virya_val}")

quick_predict("digestive stomach pain acidity Pitta")
quick_predict("stress anxiety Vata nervine adaptogen Medhya")
quick_predict("cough cold respiratory Kapha Kasahara")
quick_predict("skin disease Kushtaghna Raktashodhak")
quick_predict("Ushna virya Katu rasa deepana")

## 1️⃣3️⃣ Summary

All model artifacts are saved directly to `herbs/model/`. No download step needed!

**Next steps:**
1. Copy `.env.example` → `.env` and set your `HERB_API_KEY`
2. `pip install -r requirements.txt`
3. `uvicorn app.main:app --host 0.0.0.0 --port 8002`

In [ ]:
print(f"{'=' * 60}")
print(f"📦 All 3 artifacts saved to: {OUTPUT_DIR}/")
print(f"   • herb_model.pth    ({os.path.getsize(best_model_path)/1024/1024:.1f} MB)")
print(f"   • model_metadata.json ({os.path.getsize(meta_path)/1024/1024:.1f} MB)")
print(f"   • herb_lookup.csv   ({os.path.getsize(lookup_path)/1024/1024:.1f} MB)")
print(f"\n📊 Final accuracy: {acc_top1:.1f}% (Top-1) | {acc_top5:.1f}% (Top-5) | {acc_top10:.1f}% (Top-10)")
print(f"{'=' * 60}")
print(f"\n✅ Ready for production! Run the herb microservice with:")
print(f"   uvicorn app.main:app --host 0.0.0.0 --port 8002")